<a href="https://colab.research.google.com/github/MadhuBabuThupakula/GenAIEngineering-Cohort2/blob/main/GS_GenAI_C1_W9_S2_Multimodal_Shoe_RAG_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multimodal Shoe RAG Pipeline with Gradio App

## Import Libraries

In [ ]:
!pip install lancedb open_clip_torch
!pip install -U datasets

In [ ]:
import argparse
import os
import re
from enum import Enum
from pathlib import Path
from random import sample
from typing import Any, Dict, List, Optional

import gradio as gr
import lancedb
import pandas as pd
from datasets import load_dataset
from lancedb.embeddings import EmbeddingFunctionRegistry
from lancedb.pydantic import LanceModel, Vector
from PIL import Image
import torch
from openai import OpenAI
from transformers import AutoModelForCausalLM, AutoTokenizer


## Retriever: Vector Search and Data Management for Shoe RAG Pipeline
  
- Setting up vector embeddings using OpenAI CLIP model
- Creating and managing LanceDB vector database
- Loading data from HuggingFace datasets
- Parsing shoe attributes from text descriptions
- Performing vector similarity search on shoes

Key Concepts:
- Vector Embeddings: Converting images/text to numerical vectors for similarity search
- LanceDB: Vector database for storing and querying embeddings
- Pydantic Models: Data validation and serialization
- Similarity Search: Finding similar items based on vector distance

In [ ]:
def register_model(model_name: str) -> Any:
    """Register a model with the given name using LanceDB's EmbeddingFunctionRegistry."""
    registry = EmbeddingFunctionRegistry.get_instance()
    model = registry.get(model_name).create()
    return model


# Register the OpenAI CLIP model for vector embeddings
clip = register_model("open-clip")


In [ ]:
class MyntraShoesEnhanced(LanceModel):
    """Enhanced Myntra Shoes Schema with product metadata for vector storage."""

    vector: Vector(clip.ndims()) = clip.VectorField()
    image_uri: str = clip.SourceField()

    # Core product information extracted from text
    product_id: Optional[str] = None
    description: Optional[str] = None
    product_type: Optional[str] = None
    gender: Optional[str] = None
    color: Optional[str] = None

    # Shoe-specific attributes
    toe_shape: Optional[str] = None
    pattern: Optional[str] = None
    fastening: Optional[str] = None
    shoe_width: Optional[str] = None
    ankle_height: Optional[str] = None
    insole: Optional[str] = None
    sole_material: Optional[str] = None

    @property
    def image(self):
        if isinstance(self.image_uri, str) and os.path.exists(self.image_uri):
            return Image.open(self.image_uri)
        elif hasattr(self.image_uri, "save"):  # PIL Image object
            return self.image_uri
        else:
            # Return a placeholder or handle the case appropriately
            return None


def parse_shoe_attributes(text: str) -> dict:
    """Parse shoe attributes from the text description for structured storage."""
    attributes = {}

    # Extract product type (Men/Women + product type)
    if text.startswith("Men "):
        attributes["gender"] = "Men"
        attributes["product_type"] = text.split("Men ")[1].split(".")[0].strip()
    elif text.startswith("Women "):
        attributes["gender"] = "Women"
        attributes["product_type"] = text.split("Women ")[1].split(".")[0].strip()
    else:
        attributes["gender"] = None
        attributes["product_type"] = text.split(".")[0].strip()

    # Extract structured attributes using regex
    patterns = {
        "toe_shape": r"Toe Shape: ([^,]+)",
        "pattern": r"Pattern: ([^,]+)",
        "fastening": r"Fastening: ([^,]+)",
        "shoe_width": r"Shoe Width: ([^,]+)",
        "ankle_height": r"Ankle Height: ([^,]+)",
        "insole": r"Insole: ([^,]+)",
        "sole_material": r"Sole Material: ([^,\.]+)",
    }

    for attr, pattern in patterns.items():
        match = re.search(pattern, text)
        attributes[attr] = match.group(1).strip() if match else None

    # Extract color information (basic color detection)
    color_keywords = [
        "white",
        "black",
        "brown",
        "blue",
        "red",
        "green",
        "grey",
        "gray",
        "navy",
        "tan",
        "beige",
        "pink",
        "purple",
        "yellow",
        "orange",
    ]

    text_lower = text.lower()
    detected_colors = [color for color in color_keywords if color in text_lower]
    attributes["color"] = ", ".join(detected_colors) if detected_colors else None

    return attributes


In [ ]:
def create_shoes_table_from_hf(
    database: str,
    table_name: str,
    dataset_name: str = "Harshgarg12/myntra_shoes_dataset",
    schema: Any = MyntraShoesEnhanced,
    mode: str = "overwrite",
    sample_size: int = 500,
    save_images: bool = True,
    images_dir: str = "hf_shoe_images",
) -> None:
    """Create vector database table with shoe data from Hugging Face dataset."""

    db = lancedb.connect(database)

    if table_name in db and mode != "overwrite":
        print(f"Table {table_name} already exists")
        return

    # Load dataset from Hugging Face
    print("Loading dataset from Hugging Face...")
    ds = load_dataset(dataset_name)
    train_data = ds["train"]

    # Sample data if needed
    if len(train_data) > sample_size:
        indices = sample(range(len(train_data)), sample_size)
        train_data = train_data.select(indices)

    print(f"Processing {len(train_data)} samples...")

    # Create images directory if saving images
    if save_images:
        os.makedirs(images_dir, exist_ok=True)

    # Prepare data for table creation
    table_data = []
    for i, item in enumerate(train_data):
        image = item["image"]
        text = item["text"]

        # Parse attributes from text
        attributes = parse_shoe_attributes(text)

        # Handle image
        if save_images:
            image_path = os.path.join(images_dir, f"shoe_{i:04d}.jpg")
            image.save(image_path, "JPEG")
            image_uri = image_path
        else:
            # Store PIL image directly (may cause issues with serialization)
            image_uri = image

        table_data.append(
            {
                "image_uri": image_uri,
                "product_id": f"hf_shoe_{i:04d}",
                "description": text,
                "product_type": attributes.get("product_type"),
                "gender": attributes.get("gender"),
                "color": attributes.get("color"),
                "toe_shape": attributes.get("toe_shape"),
                "pattern": attributes.get("pattern"),
                "fastening": attributes.get("fastening"),
                "shoe_width": attributes.get("shoe_width"),
                "ankle_height": attributes.get("ankle_height"),
                "insole": attributes.get("insole"),
                "sole_material": attributes.get("sole_material"),
            }
        )

    if table_data:
        if table_name in db:
            db.drop_table(table_name)

        table = db.create_table(table_name, schema=schema, mode="create")
        table.add(pd.DataFrame(table_data))
        print(f"Added {len(table_data)} shoes to table")
    else:
        print("No data to add")


In [ ]:
def run_shoes_search(
    database: str,
    table_name: str,
    schema: Any,
    search_query: Any,
    limit: int = 6,
    output_folder: str = "output_retriever",
    search_type: str = "auto",  # "auto", "text", "image"
) -> tuple[list, str]:
    """RETRIEVAL: Run vector search on shoes and return detailed results."""

    # Clean output folder
    if os.path.exists(output_folder):
        for file in os.listdir(output_folder):
            os.remove(os.path.join(output_folder, file))
    else:
        os.makedirs(output_folder)

    db = lancedb.connect(database)
    table = db.open_table(table_name)

    # Determine search type and process query
    actual_search_type = search_type
    processed_query = search_query

    if search_type == "auto":
        # Auto-detect search type
        if isinstance(search_query, str):
            if search_query.endswith((".jpg", ".jpeg", ".png", ".bmp", ".gif")):
                # Image file path
                try:
                    processed_query = Image.open(search_query)
                    actual_search_type = "image"
                    print(f"🖼️  Detected image search: {search_query}")
                except Exception as e:
                    print(f"❌ Error loading image: {e}")
                    return [], "error"
            else:
                # Text query
                actual_search_type = "text"
                print(f"📝 Detected text search: {search_query}")
        elif hasattr(search_query, "save"):  # PIL Image object
            actual_search_type = "image"
            processed_query = search_query
            print("🖼️  Detected image search: PIL Image object")
        else:
            actual_search_type = "text"
            print(f"📝 Detected text search: {search_query}")

    elif search_type == "image":
        if isinstance(search_query, str):
            try:
                processed_query = Image.open(search_query)
                print(f"🖼️  Image search: {search_query}")
            except Exception as e:
                print(f"❌ Error loading image: {e}")
                return [], "error"
        elif hasattr(search_query, "save"):
            processed_query = search_query
            print("🖼️  Image search: PIL Image object")
        else:
            print("❌ Invalid image input for image search")
            return [], "error"

    else:  # text search
        actual_search_type = "text"
        print(f"📝 Text search: {search_query}")

    # Perform vector search
    try:
        results = table.search(processed_query).limit(limit).to_pydantic(schema)
    except Exception as e:
        print(f"❌ Search error: {e}")
        return [], "error"

    # Save images and collect metadata
    search_results = []
    for i, result in enumerate(results):
        image_path = os.path.join(output_folder, f"result_{i}.jpg")

        # Handle different image storage methods
        if result.image:
            result.image.save(image_path, "JPEG")
        else:
            print(f"Warning: No image available for result {i}")
            continue

        search_results.append(
            {
                "rank": i + 1,
                "product_id": result.product_id,
                "description": result.description,  # Remove truncation
                "product_type": result.product_type,
                "gender": result.gender,
                "color": result.color,
                "toe_shape": result.toe_shape,
                "pattern": result.pattern,
                "fastening": result.fastening,
                "image_path": image_path,
            }
        )

    return search_results, actual_search_type


---

## Augmenter - Context Enhancement and Prompt Engineering for Shoe RAG Pipeline

- Query classification and analysis
- Context formatting and enhancement
- Prompt engineering with different strategies
- Advanced prompt templates for different query types

Key Concepts:
- Prompt Engineering: Designing effective prompts to guide LLM responses
- Context Enhancement: Structuring retrieved data for optimal LLM understanding
- Query Classification: Determining intent to apply appropriate prompt strategies
- Template-based Prompting: Using structured templates for consistent results

In [ ]:
class QueryType(Enum):
    """Query types for different shoe-related interactions."""

    RECOMMENDATION = "recommendation"
    SEARCH = "search"

In [ ]:
class SimpleShoePrompts:
    """AUGMENTATION: Simplified prompt system for shoe RAG with context enhancement."""

    def __init__(self):
        self.system_prompts = {
            "recommendation": """You are a helpful assistant. Choose from the given shoe options and give a short, simple recommendation. Do not make up any information.""",
            "search": """You are a knowledgeable shoe assistant. Help customers understand the available shoe options
that match their search criteria, providing detailed information about features and benefits.""",
        }

    def classify_query(self, query: str) -> QueryType:
        """Classify query into recommendation or search type."""
        query_lower = query.lower()

        if any(
            word in query_lower
            for word in ["recommend", "suggest", "best", "need", "looking for"]
        ):
            return QueryType.RECOMMENDATION
        else:
            return QueryType.SEARCH

    def format_shoes_context(self, shoes: List[Dict[str, Any]]) -> str:
        """AUGMENTATION: Format retrieved shoes into readable context for LLM."""
        formatted_shoes = []
        for i, shoe in enumerate(shoes, 1):
            # Keep it simple - just basic info
            product_type = shoe.get("product_type", "Shoe")
            gender = shoe.get("gender", "")

            if gender:
                shoe_name = f"{product_type} for {gender}"
            else:
                shoe_name = product_type

            # Add basic color info if available
            color = shoe.get("color", "")
            if color and color not in ["None", None, ""]:
                shoe_name += f" ({color})"

            formatted_shoes.append(f"{i}. {shoe_name}")

        return "\n".join(formatted_shoes)

    def generate_prompt(
        self, query: str, shoes: List[Dict[str, Any]], search_type: str = "text"
    ) -> str:
        """AUGMENTATION: Generate complete prompt based on query type and retrieved context."""
        # If it's an image search, always treat as search query type
        if search_type == "image":
            query_type = QueryType.SEARCH
        else:
            query_type = self.classify_query(query)

        system_prompt = self.system_prompts[query_type.value]
        context = self.format_shoes_context(shoes)

        if query_type == QueryType.RECOMMENDATION:
            # Add a summary to guide recommendations
            intent_summary = (
                f"Based on the query, the user is likely looking for {query.lower()}."
            )

            user_prompt = f"""{intent_summary}

Available Options:
{context}

Your task:
- Recommend the best option(s) that align most closely with the query.
- Reference specific attributes (e.g., gender, product type, color, or other features) in your reasoning.
- Avoid adding details not provided in the context.

Provide your recommendation in 2-3 sentences."""

        else:  # SEARCH
            user_prompt = f"""Here are shoes matching: "{query}"

Search Results:
{context}

Explain how well these shoes meet the search criteria and highlight their relevant features."""

        return f"{system_prompt}\n\n{user_prompt}"

In [ ]:
def detect_search_type(search_query) -> str:
    """Auto-detect search type based on query content (matches retriever.py logic)."""
    # Auto-detect search type
    if isinstance(search_query, str):
        if search_query.endswith((".jpg", ".jpeg", ".png", ".bmp", ".gif")):
            # Image file path
            return "image"
        else:
            # Text query
            return "text"
    elif hasattr(search_query, "save"):  # PIL Image object
        return "image"
    else:
        return "text"


def get_real_shoes_data(
    query: str,
    search_type: str = "text",
    database: str = "myntra_shoes_db",
    table_name: str = "myntra_shoes_table",
    limit: int = 3,
) -> List[Dict[str, Any]]:
    """Get real shoes data from retriever for testing purposes."""

    try:
        results, _ = run_shoes_search(
            database=database,
            table_name=table_name,
            schema=MyntraShoesEnhanced,
            search_query=query,
            limit=limit,
            search_type=search_type,
            output_folder="output_augmenter",
        )
        return results
    except Exception as e:
        raise Exception(
            f"Could not retrieve real data: {e}. Please ensure the database is set up correctly."
        )

---


## Generator - LLM Setup and Response Generation for Shoe RAG Pipeline

- Setting up different LLM providers (Qwen, OpenAI)
- Managing model configurations and parameters
- Generating responses using augmented context
- Handling different model types and API integrations

Key Concepts:
- Large Language Models (LLMs): AI models that generate human-like text
- Model Providers: Different services/frameworks for running LLMs
- API Integration: Connecting to external services like OpenAI
- Local Models: Running models locally with transformers
- Generation Parameters: Temperature, max tokens, etc. for controlling output

In [ ]:
def setup_qwen_model(model_name: str = "Qwen/Qwen2.5-0.5B-Instruct") -> tuple:
    """GENERATION: Setup Qwen2.5-0.5B model for text generation."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.float32, device_map="cpu"
    )
    return tokenizer, model


def setup_openai_client(api_key: str) -> OpenAI:
    """GENERATION: Setup OpenAI client for text generation."""
    if not api_key or api_key.strip() == "":
        raise ValueError("OpenAI API key is required")

    client = OpenAI(api_key=api_key)
    return client


def get_available_models() -> Dict[str, List[str]]:
    """Get available models for each provider."""
    models = {
        "qwen": [
            "Qwen/Qwen2.5-0.5B-Instruct",
            "Qwen/Qwen2.5-1.5B-Instruct",
            "Qwen/Qwen2.5-3B-Instruct",
            "Qwen/Qwen2.5-7B-Instruct",
        ],
        "openai": ["gpt-4o-mini", "gpt-4o", "gpt-4-turbo", "gpt-3.5-turbo"],
    }
    return models


In [ ]:
def generate_shoes_rag_response(
    query: str,
    retrieved_shoes: List[Dict[str, any]],
    model_provider: str = "qwen",
    model_name: str = "Qwen/Qwen2.5-0.5B-Instruct",
    openai_client: Optional[OpenAI] = None,
    tokenizer=None,
    model=None,
    max_tokens: int = 200,
    use_advanced_prompts: bool = True,
) -> str:
    """GENERATION: Generate RAG response using retrieved shoes context with prompt engineering."""

    if use_advanced_prompts:
        # Use the simplified prompt system

        prompt_manager = SimpleShoePrompts()
        complete_prompt = prompt_manager.generate_prompt(query, retrieved_shoes)
        query_type = prompt_manager.classify_query(query)
        print(f"Using {query_type.value} prompt for query")

    else:
        # Use the basic prompt system (fallback)

        prompt_manager = SimpleShoePrompts()
        context = prompt_manager.format_shoes_context(retrieved_shoes)
        complete_prompt = f"""Based on the following shoe products, answer the user's question:

Shoes:
{context}

Question: {query}

Answer:"""

    if model_provider == "openai":
        if not openai_client:
            raise ValueError("OpenAI client is required for OpenAI models")

        try:
            response = openai_client.chat.completions.create(
                model=model_name,
                messages=[
                    {
                        "role": "system",
                        "content": "You are a helpful shoe recommendation assistant.",
                    },
                    {"role": "user", "content": complete_prompt},
                ],
                max_tokens=max_tokens,
                temperature=0.1,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            raise Exception(f"OpenAI API error: {str(e)}")

    else:  # Qwen model
        if not tokenizer or not model:
            raise ValueError("Tokenizer and model are required for Qwen models")

        inputs = tokenizer(
            complete_prompt, return_tensors="pt", truncation=True, max_length=2048
        )

        # Ensure everything runs on CPU
        inputs = {k: v.to("cpu") for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                temperature=0.1,  # Very low temperature to reduce hallucination
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.05,  # Minimal repetition penalty
                no_repeat_ngram_size=2,
                early_stopping=False,
            )

        response = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
        )
        return response.strip()


---

## End to End RAG Pipeline for Shoe Search and Recommendation

This script integrates all three phases of the RAG pipeline:
1. RETRIEVAL: Vector search and data management (from retriever.py)
2. AUGMENTATION: Context enhancement and prompt engineering (from augmenter.py)
3. GENERATION: LLM setup and response generation (from generator.py)

Key Concepts:
- RAG (Retrieval-Augmented Generation): A technique that combines information retrieval
  with language generation to provide accurate, contextual responses
- Pipeline Integration: Connecting multiple AI components in sequence
- End-to-End Processing: Complete workflow from query to final response
- Multi-modal Search: Supporting both text and image queries

In [ ]:
def run_complete_shoes_rag_pipeline(
    database: str,
    table_name: str,
    schema: Any,
    search_query: Any,  # Can be text string or image path/PIL Image
    limit: int = 3,
    use_llm: bool = True,
    use_advanced_prompts: bool = True,
    search_type: str = "auto",
    model_provider: str = "qwen",
    model_name: str = "Qwen/Qwen2.5-0.5B-Instruct",
    openai_api_key: Optional[str] = None,
) -> Dict[str, Any]:
    """Run complete RAG pipeline integrating Retrieval, Augmentation, and Generation."""

    # SECTION 1: RETRIEVAL - Get relevant shoes from vector database
    print("🔍 RETRIEVAL: Searching for relevant shoes...")
    results, actual_search_type = run_shoes_search(
        database, table_name, schema, search_query, limit, search_type=search_type
    )

    if not results:
        return {
            "query": search_query,
            "results": [],
            "response": "No results found",
            "search_type": actual_search_type,
        }

    if not use_llm:
        return {
            "query": search_query,
            "results": results,
            "response": None,
            "search_type": actual_search_type,
        }

    # SECTION 2: AUGMENTATION - Process and enhance context with prompt engineering
    try:
        print("📝 AUGMENTATION: Enhancing context with prompt engineering...")

        # Set up prompt manager and analyze query
        prompt_manager = SimpleShoePrompts()

        # For image search, use appropriate query text
        if actual_search_type == "image":
            query_text = "similar shoes based on the provided image"
            print(f"   └─ Image search - using search query type")
        else:
            query_text = str(search_query)
            query_type = prompt_manager.classify_query(query_text)
            print(f"   └─ Text query classified as: {query_type.value}")

        # Format context and generate enhanced prompt
        enhanced_prompt = prompt_manager.generate_prompt(
            query_text, results, actual_search_type
        )
        print(f"   └─ Context formatted with {len(results)} retrieved shoes")

        # SECTION 3: GENERATION - Setup LLM and generate response
        print("🤖 GENERATION: Setting up LLM and generating response...")

        tokenizer, model, openai_client = None, None, None

        if model_provider == "openai":
            if not openai_api_key:
                raise ValueError("OpenAI API key is required for OpenAI models")
            openai_client = setup_openai_client(openai_api_key)
            print(f"   └─ OpenAI client setup with model: {model_name}")
        else:
            tokenizer, model = setup_qwen_model(model_name)
            print(f"   └─ Qwen model loaded: {model_name}")

        # Generate final response using augmented context
        response = generate_shoes_rag_response(
            query=query_text,
            retrieved_shoes=results,
            model_provider=model_provider,
            model_name=model_name,
            openai_client=openai_client,
            tokenizer=tokenizer,
            model=model,
            max_tokens=200,
            use_advanced_prompts=use_advanced_prompts,
        )

        # Add prompt analysis
        if actual_search_type == "image":
            final_query_type = QueryType.SEARCH.value
        else:
            final_query_type = query_type.value

        prompt_analysis = {
            "query_type": final_query_type,
            "num_results": len(results),
            "search_type": actual_search_type,
        }

        return {
            "query": search_query,
            "results": results,
            "response": response,
            "prompt_analysis": prompt_analysis,
            "search_type": actual_search_type,
        }
    except Exception as e:
        print(f"LLM generation failed: {e}")
        return {
            "query": search_query,
            "results": results,
            "response": "LLM unavailable - showing search results only",
            "search_type": actual_search_type,
        }


In [ ]:
def run_complete_shoes_rag_pipeline_with_details(
    database: str,
    table_name: str,
    schema: Any,
    search_query: Any,  # Can be text string or image path/PIL Image
    limit: int = 3,
    use_llm: bool = True,
    use_advanced_prompts: bool = True,
    search_type: str = "auto",
    model_provider: str = "qwen",
    model_name: str = "Qwen/Qwen2.5-0.5B-Instruct",
    openai_api_key: Optional[str] = None,
) -> Dict[str, Any]:
    """Run complete RAG pipeline with detailed step tracking."""

    # Initialize step details
    retrieval_details = ""
    augmentation_details = ""
    generation_details = ""

    # SECTION 1: RETRIEVAL - Get relevant shoes from vector database
    retrieval_details += "🔍 RETRIEVAL PHASE\n"
    retrieval_details += "=" * 50 + "\n"
    retrieval_details += f"🎯 Query Type: {search_type}\n"
    retrieval_details += f"🔍 Searching vector database...\n"

    results, actual_search_type = run_shoes_search(
        database, table_name, schema, search_query, limit, search_type=search_type
    )

    retrieval_details += f"✅ Search completed!\n"
    retrieval_details += f"📊 Search Type Detected: {actual_search_type}\n"
    retrieval_details += f"📈 Results Found: {len(results)}\n\n"

    if results:
        retrieval_details += "🎯 Retrieved Products:\n"
        for i, result in enumerate(results, 1):
            retrieval_details += f"  {i}. {result.get('product_type', 'Shoe')} for {result.get('gender', 'Unisex')}\n"
            retrieval_details += f"     Color: {result.get('color', 'N/A')}\n"
            retrieval_details += f"     Pattern: {result.get('pattern', 'N/A')}\n"
            if result.get("description"):
                # Show full description without truncation
                retrieval_details += f"     Description: {result['description']}\n"
            retrieval_details += "\n"
    else:
        retrieval_details += "❌ No results found\n"
        return {
            "query": search_query,
            "results": [],
            "response": "No results found",
            "search_type": actual_search_type,
            "retrieval_details": retrieval_details,
            "augmentation_details": "⏭️ Skipped - No results to process",
            "generation_details": "⏭️ Skipped - No results to process",
        }

    if not use_llm:
        return {
            "query": search_query,
            "results": results,
            "response": None,
            "search_type": actual_search_type,
            "retrieval_details": retrieval_details,
            "augmentation_details": "⏭️ Skipped - LLM disabled",
            "generation_details": "⏭️ Skipped - LLM disabled",
        }

    # SECTION 2: AUGMENTATION - Process and enhance context with prompt engineering
    try:
        augmentation_details += "📝 AUGMENTATION PHASE\n"
        augmentation_details += "=" * 50 + "\n"

        # Set up prompt manager and analyze query
        prompt_manager = SimpleShoePrompts()

        # For image search, use appropriate query text
        if actual_search_type == "image":
            query_text = "similar shoes based on the provided image"
            augmentation_details += f"🖼️ Image Search Detected\n"
            augmentation_details += f"🔄 Query Text: '{query_text}'\n"
        else:
            query_text = str(search_query)
            query_type = prompt_manager.classify_query(query_text)
            augmentation_details += f"📝 Text Query: '{query_text}'\n"
            augmentation_details += f"🎯 Query Classification: {query_type.value}\n"

        # Format context and generate enhanced prompt
        enhanced_prompt = prompt_manager.generate_prompt(
            query_text, results, actual_search_type
        )

        augmentation_details += f"📊 Context Processing:\n"
        augmentation_details += f"  • Products formatted: {len(results)}\n"
        augmentation_details += (
            f"  • Prompt strategy: {'Advanced' if use_advanced_prompts else 'Basic'}\n"
        )
        augmentation_details += (
            f"  • Prompt length: {len(enhanced_prompt)} characters\n\n"
        )

        # Show the full prompt instead of preview
        augmentation_details += f"🔍 Full Prompt:\n{enhanced_prompt}\n\n"

        # SECTION 3: GENERATION - Setup LLM and generate response
        generation_details += "🤖 GENERATION PHASE\n"
        generation_details += "=" * 50 + "\n"
        generation_details += f"🏭 Model Provider: {model_provider}\n"
        generation_details += f"🎯 Model Name: {model_name}\n"

        tokenizer, model, openai_client = None, None, None

        if model_provider == "openai":
            if not openai_api_key:
                raise ValueError("OpenAI API key is required for OpenAI models")
            openai_client = setup_openai_client(openai_api_key)
            generation_details += f"✅ OpenAI client initialized\n"
            generation_details += f"🔑 API key: {'*' * (len(openai_api_key) - 8) + openai_api_key[-4:] if len(openai_api_key) > 8 else '****'}\n"
        else:
            tokenizer, model = setup_qwen_model(model_name)
            generation_details += f"✅ Qwen model loaded\n"
            generation_details += f"💾 Model size: {model_name}\n"

        generation_details += f"⚙️ Generation settings:\n"
        generation_details += f"  • Max tokens: 200\n"
        generation_details += f"  • Temperature: 0.1 (low for consistency)\n"
        generation_details += f"  • Advanced prompts: {use_advanced_prompts}\n\n"

        generation_details += f"🔄 Generating response...\n"

        # Generate final response using augmented context
        response = generate_shoes_rag_response(
            query=query_text,
            retrieved_shoes=results,
            model_provider=model_provider,
            model_name=model_name,
            openai_client=openai_client,
            tokenizer=tokenizer,
            model=model,
            max_tokens=200,
            use_advanced_prompts=use_advanced_prompts,
        )

        generation_details += f"✅ Response generated!\n"
        generation_details += f"📏 Response length: {len(response)} characters\n"
        generation_details += f"📝 Full Response:\n{response}\n"

        # Add prompt analysis
        if actual_search_type == "image":
            final_query_type = QueryType.SEARCH.value
        else:
            final_query_type = query_type.value

        prompt_analysis = {
            "query_type": final_query_type,
            "num_results": len(results),
            "search_type": actual_search_type,
        }

        return {
            "query": search_query,
            "results": results,
            "response": response,
            "prompt_analysis": prompt_analysis,
            "search_type": actual_search_type,
            "retrieval_details": retrieval_details,
            "augmentation_details": augmentation_details,
            "generation_details": generation_details,
        }
    except Exception as e:
        error_msg = f"❌ LLM generation failed: {str(e)}"
        generation_details += error_msg
        return {
            "query": search_query,
            "results": results,
            "response": "LLM unavailable - showing search results only",
            "search_type": actual_search_type,
            "retrieval_details": retrieval_details,
            "augmentation_details": augmentation_details,
            "generation_details": generation_details,
        }


---

## Gradio App: Gradio Web Interface for Shoe RAG Pipeline

The following code blocks provides a web-based interface for the complete RAG pipeline using Gradio.
It integrates all components (retrieval, augmentation, generation) into a user-friendly web app.

Key Concepts:
- Gradio: Python library for creating web interfaces for ML models
- Web Interface: User-friendly way to interact with RAG pipeline
- Multimodal Input: Supporting both text and image inputs
- Real-time Processing: Live interaction with the RAG system
- Step-by-step Visualization: Showing detailed pipeline execution

In [ ]:
def gradio_rag_pipeline(
    query,
    image,
    search_type,
    use_advanced_prompts,
    model_provider,
    model_name,
    openai_api_key,
):
    """Gradio interface function for RAG pipeline."""
    try:
        # Validate inputs based on model provider
        if model_provider == "openai":
            if not openai_api_key or openai_api_key.strip() == "":
                return (
                    "❌ OpenAI API key is required for OpenAI models.",
                    "",
                    "",
                    "",
                    "",
                    [],
                )

        # Check if both text and image inputs are provided - this is not allowed
        has_text_input = query and query.strip()
        has_image_input = image is not None

        if has_text_input and has_image_input:
            return (
                "❌ Error: Please provide either a text query OR an image, not both. Choose one input type at a time.",
                "",
                "",
                "",
                "",
                [],
            )

        # Determine the actual query based on inputs
        if search_type == "image" and image is not None:
            actual_query = image
        elif search_type == "text" and query.strip():
            actual_query = query
        elif search_type == "auto":
            if image is not None:
                actual_query = image
            elif query.strip():
                actual_query = query
            else:
                return (
                    "❌ Please provide either a text query or upload an image.",
                    "",
                    "",
                    "",
                    "",
                    [],
                )
        else:
            return (
                "❌ Please provide appropriate input for the selected search type.",
                "",
                "",
                "",
                "",
                [],
            )

        # Run the RAG pipeline with detailed tracking
        rag_result = run_complete_shoes_rag_pipeline_with_details(
            database="myntra_shoes_db",
            table_name="myntra_shoes_table",
            schema=MyntraShoesEnhanced,
            search_query=actual_query,
            limit=3,
            use_llm=True,
            use_advanced_prompts=use_advanced_prompts,
            search_type=search_type,
            model_provider=model_provider,
            model_name=model_name,
            openai_api_key=openai_api_key if model_provider == "openai" else None,
        )

        # Extract detailed step information
        retrieval_details = rag_result.get(
            "retrieval_details", "No retrieval details available"
        )
        augmentation_details = rag_result.get(
            "augmentation_details", "No augmentation details available"
        )
        generation_details = rag_result.get(
            "generation_details", "No generation details available"
        )

        # Format the response
        response = rag_result.get("response", "No response generated")
        search_type_used = rag_result.get("search_type", "unknown")

        # Format results for display
        results_text = f"🔍 Search Type: {search_type_used}\n\n"
        if rag_result.get("prompt_analysis"):
            results_text += (
                f"📝 Query Type: {rag_result['prompt_analysis']['query_type']}\n"
            )
            results_text += (
                f"📊 Results Found: {rag_result['prompt_analysis']['num_results']}\n\n"
            )

        # Prepare image gallery data
        image_gallery = []
        results_details = []

        for i, result in enumerate(rag_result["results"], 1):
            product_type = result.get("product_type", "Shoe")
            gender = result.get("gender", "Unisex")
            color = result.get("color", "Various colors")
            pattern = result.get("pattern", "Standard")
            description = result.get("description", "No description available")
            image_path = result.get("image_path")

            # Add to gallery if image exists
            if image_path and os.path.exists(image_path):
                # Create detailed caption for the image
                caption = f"#{i} - {product_type} for {gender}"
                if color and color not in ["None", None, ""]:
                    caption += f" | Color: {color}"
                if pattern and pattern not in ["None", None, ""]:
                    caption += f" | Pattern: {pattern}"

                image_gallery.append((image_path, caption))

            # Format detailed description
            detail_text = f"**{i}. {product_type} for {gender}**\n"
            detail_text += f"   • Color: {color}\n"
            detail_text += f"   • Pattern: {pattern}\n"
            if description:
                # Show full description without truncation
                detail_text += f"   • Description: {description}\n"
            detail_text += "\n"
            results_details.append(detail_text)

        # Combine all details
        formatted_results = "".join(results_details)

        return (
            response,
            formatted_results,
            retrieval_details,
            augmentation_details,
            generation_details,
            image_gallery,
        )

    except Exception as e:
        return (
            f"❌ Error: {str(e)}",
            "",
            "❌ Error occurred",
            "❌ Error occurred",
            "❌ Error occurred",
            [],
        )


In [ ]:
def create_gradio_app():
    """Create and launch the Gradio application."""

    # Custom CSS for better styling
    css = """
    .gradio-container {
        max-width: 100% !important;
        width: 100% !important;
        margin: 0 !important;
        padding: 20px !important;
        background: #f5f5f5;
    }
    .main {
        max-width: 100% !important;
        width: 100% !important;
    }
    /* Header styling */
    .header-section {
        text-align: center;
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        color: white;
        padding: 30px;
        border-radius: 15px;
        margin-bottom: 30px;
        box-shadow: 0 4px 15px rgba(0,0,0,0.1);
    }
    .header-section h1 {
        font-size: 2.5em;
        margin-bottom: 15px;
        text-shadow: 2px 2px 4px rgba(0,0,0,0.3);
    }
    .header-section p {
        font-size: 1.2em;
        margin-bottom: 10px;
        opacity: 0.95;
    }
    .output-text {
        font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
        line-height: 1.6;
        color: #333;
    }
    .gallery-container {
        margin-top: 15px;
        width: 100%;
    }
    .search-section {
        background: #ffffff;
        padding: 25px;
        border-radius: 12px;
        margin-bottom: 20px;
        height: fit-content;
        box-shadow: 0 2px 10px rgba(0,0,0,0.08);
        border: 1px solid #e0e0e0;
    }
    .results-section {
        background: #ffffff;
        padding: 25px;
        border-radius: 12px;
        border: 1px solid #e0e0e0;
        height: fit-content;
        box-shadow: 0 2px 10px rgba(0,0,0,0.08);
    }
    .gallery-section {
        background: #ffffff;
        padding: 25px;
        border-radius: 12px;
        border: 1px solid #e0e0e0;
        margin-top: 20px;
        width: 100%;
        box-shadow: 0 2px 10px rgba(0,0,0,0.08);
    }
    /* Improve text readability */
    .gradio-textbox textarea {
        background: #fafafa !important;
        border: 1px solid #ddd !important;
        color: #333 !important;
    }
    .gradio-textbox textarea:focus {
        background: #ffffff !important;
        border-color: #667eea !important;
    }
    /* Make gallery images larger */
    .gallery img {
        max-height: 300px !important;
        object-fit: contain !important;
    }
    /* Improve button styling */
    .primary-button {
        width: 100%;
        padding: 12px;
        font-size: 16px;
        font-weight: bold;
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important;
        border: none !important;
        box-shadow: 0 4px 15px rgba(102, 126, 234, 0.4) !important;
    }
    .primary-button:hover {
        transform: translateY(-2px);
        box-shadow: 0 6px 20px rgba(102, 126, 234, 0.6) !important;
    }
    /* Section headers */
    .section-header {
        color: #667eea;
        font-weight: bold;
        border-bottom: 2px solid #667eea;
        padding-bottom: 5px;
        margin-bottom: 15px;
    }
    /* Model settings styling */
    .model-settings {
        background: #f8f9ff;
        padding: 15px;
        border-radius: 8px;
        margin: 10px 0;
        border: 1px solid #e0e7ff;
    }
    /* API key input styling */
    .api-key-input {
        border: 2px solid #fbbf24 !important;
        background: #fffbeb !important;
    }
    .api-key-input:focus {
        border-color: #f59e0b !important;
        box-shadow: 0 0 0 3px rgba(245, 158, 11, 0.1) !important;
    }
    /* RAG steps styling */
    .rag-steps-section {
        background: linear-gradient(135deg, #f8fafc 0%, #f1f5f9 100%);
        padding: 25px;
        border-radius: 12px;
        margin: 20px 0;
        border: 1px solid #e2e8f0;
    }
    .step-box {
        background: #ffffff;
        border: 2px solid #e5e7eb;
        border-radius: 10px;
        padding: 20px;
        margin: 10px 0;
        transition: all 0.3s ease;
    }
    .step-box:hover {
        border-color: #667eea;
        box-shadow: 0 4px 12px rgba(102, 126, 234, 0.15);
    }
    /* Different colors for each step */
    .retrieval-step {
        border-left: 4px solid #10b981;
    }
    .augmentation-step {
        border-left: 4px solid #3b82f6;
    }
    .generation-step {
        border-left: 4px solid #8b5cf6;
    }
    """

    with gr.Blocks(css=css, title="👟 Shoe RAG Pipeline") as app:
        # Header Section
        with gr.Row():
            with gr.Column(elem_classes=["header-section"]):
                gr.HTML(
                    """
                <div style="text-align: center;">
                    <h1>👟 Multimodal Shoe RAG Pipeline</h1>
                    <p>This demo showcases a complete <strong>Retrieval-Augmented Generation (RAG)</strong> pipeline for shoe recommendations and search.</p>
                    <div style="display: flex; justify-content: center; gap: 25px; margin-top: 20px; flex-wrap: wrap;">
                        <div>🔍 <strong>Text Search</strong><br/>Natural language queries</div>
                        <div>🖼️ <strong>Image Search</strong><br/>Visual similarity matching</div>
                        <div>🤖 <strong>AI Models</strong><br/>Qwen & OpenAI support</div>
                        <div>🔐 <strong>Secure API</strong><br/>Protected key handling</div>
                        <div>📊 <strong>Structured Results</strong><br/>Detailed product information</div>
                    </div>
                </div>
                """
                )

        with gr.Row(equal_height=False):
            # Left Column - Search Input
            with gr.Column(scale=1, elem_classes=["search-section"]):
                gr.HTML('<h3 class="section-header">🔍 Search Input</h3>')

                query = gr.Textbox(
                    label="Text Query",
                    placeholder="e.g., 'Recommend running shoes for men' or 'Show me casual sneakers'",
                    lines=4,
                    max_lines=6,
                )

                image = gr.Image(
                    label="Upload Shoe Image (for image search)", type="pil", height=220
                )

                with gr.Row():
                    search_type = gr.Radio(
                        choices=["auto", "text", "image"],
                        value="auto",
                        label="Search Type",
                        info="Auto-detect or force specific search type",
                    )

                gr.HTML('<h3 class="section-header">🤖 AI Model Settings</h3>')

                # Model provider selection
                provider_choices = ["qwen", "openai"]

                model_provider = gr.Radio(
                    choices=provider_choices,
                    value="qwen",
                    label="Model Provider",
                    info="Choose between local Qwen models or OpenAI API"
                )

                # Model selection dropdown - will be updated based on provider
                available_models = get_available_models()
                model_name = gr.Dropdown(
                    choices=available_models["qwen"],
                    value="Qwen/Qwen2.5-0.5B-Instruct",
                    label="Model Name",
                    info="Select the specific model to use",
                )

                # OpenAI API Key input (hidden by default)
                openai_api_key = gr.Textbox(
                    label="OpenAI API Key",
                    placeholder="Enter your OpenAI API key (required for OpenAI models)",
                    type="password",
                    visible=False,
                    info="Your API key is secure and not stored",
                )

                use_advanced_prompts = gr.Checkbox(
                    value=True,
                    label="Use Advanced Prompts",
                    info="Enable enhanced prompt engineering for better responses",
                )

                search_btn = gr.Button(
                    "🔍 Search",
                    variant="primary",
                    size="lg",
                    elem_classes=["primary-button"],
                )

                # JavaScript to update model choices and show/hide API key based on provider
                def update_model_choices(provider):
                    models = get_available_models()
                    choices = models[provider]
                    default_value = choices[0]
                    api_key_visible = provider == "openai"
                    return gr.Dropdown(
                        choices=choices, value=default_value
                    ), gr.Textbox(visible=api_key_visible)

                model_provider.change(
                    fn=update_model_choices,
                    inputs=[model_provider],
                    outputs=[model_name, openai_api_key],
                )

            # Right Column - Results
            with gr.Column(scale=2, elem_classes=["results-section"]):
                gr.HTML('<h3 class="section-header">🤖 Final AI Response</h3>')
                response_output = gr.Textbox(
                    label="RAG Response",
                    lines=6,
                    max_lines=20,  # Increased max lines
                    elem_classes=["output-text"],
                    show_copy_button=True,
                )

                gr.HTML('<h3 class="section-header">📊 Product Information</h3>')
                results_output = gr.Textbox(
                    label="Retrieved Products Summary",
                    lines=8,
                    max_lines=25,  # Increased max lines
                    elem_classes=["output-text"],
                    show_copy_button=True,
                )

        # Full width section for image gallery
        with gr.Row():
            with gr.Column(elem_classes=["gallery-section"]):
                gr.HTML('<h3 class="section-header">🖼️ Retrieved Shoe Images</h3>')
                image_gallery = gr.Gallery(
                    label="Search Results Gallery",
                    show_label=False,
                    elem_id="gallery",
                    columns=3,
                    rows=1,
                    object_fit="contain",
                    height=350,
                    elem_classes=["gallery-container"],
                    preview=True,
                )

        # RAG Pipeline Steps - Three columns for detailed breakdown
        gr.HTML(
            '<h2 style="text-align: center; color: #667eea; margin: 30px 0 20px 0;">🔍 RAG Pipeline Step-by-Step Breakdown</h2>'
        )

        with gr.Row(equal_height=True, elem_classes=["rag-steps-section"]):
            # Step 1: Retrieval
            with gr.Column(scale=1, elem_classes=["step-box", "retrieval-step"]):
                gr.HTML('<h3 class="section-header">🔍 Step 1: Retrieval</h3>')
                retrieval_output = gr.Textbox(
                    label="Vector Search & Data Retrieval",
                    lines=15,
                    max_lines=30,  # Increased max lines
                    elem_classes=["output-text"],
                    show_copy_button=True,
                    info="Details about vector search, similarity matching, and retrieved products",
                )

            # Step 2: Augmentation
            with gr.Column(scale=1, elem_classes=["step-box", "augmentation-step"]):
                gr.HTML('<h3 class="section-header">📝 Step 2: Augmentation</h3>')
                augmentation_output = gr.Textbox(
                    label="Context Enhancement & Prompt Engineering",
                    lines=15,
                    max_lines=30,  # Increased max lines
                    elem_classes=["output-text"],
                    show_copy_button=True,
                    info="Query analysis, context formatting, and prompt construction",
                )

            # Step 3: Generation
            with gr.Column(scale=1, elem_classes=["step-box", "generation-step"]):
                gr.HTML('<h3 class="section-header">🤖 Step 3: Generation</h3>')
                generation_output = gr.Textbox(
                    label="LLM Response Generation",
                    lines=15,
                    max_lines=30,  # Increased max lines
                    elem_classes=["output-text"],
                    show_copy_button=True,
                    info="Model setup, generation parameters, and response creation",
                )

        # Event handlers
        search_btn.click(
            fn=gradio_rag_pipeline,
            inputs=[
                query,
                image,
                search_type,
                use_advanced_prompts,
                model_provider,
                model_name,
                openai_api_key,
            ],
            outputs=[
                response_output,
                results_output,
                retrieval_output,
                augmentation_output,
                generation_output,
                image_gallery,
            ],
        )

    return app


In [ ]:
print("🔄 Setting up database from HuggingFace dataset...")
create_shoes_table_from_hf(
    database="myntra_shoes_db",
    table_name="myntra_shoes_table",
    sample_size=500,
    save_images=True,
)
print("✅ Database setup complete!")

100%|██████████| 64/64 [00:20<00:00,  3.17it/s]


🔄 Setting up database from HuggingFace dataset...
Loading dataset from Hugging Face...


  0%|          | 0/64 [00:00<?, ?it/s]

Processing 50 samples...


100%|██████████| 64/64 [00:30<00:00,  2.07it/s]

100%|██████████| 50/50 [00:27<00:00,  1.82it/s]

Added 50 shoes to table
✅ Database setup complete!


In [ ]:
print("Initializing Gradio App")
app = create_gradio_app()
print("✅ Gradio App initialized")

Initializing Gradio App


 44%|████▍     | 28/64 [00:08<00:08,  4.06it/s]

✅ Gradio App initialized


In [ ]:
app.launch(
        share=True,
        server_name="127.0.0.1",
        server_port=7866,
        show_error=True,
        debug=True
    )

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1121f9bb675ff7c862.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


📝 Detected text search: recommend shoes
Using recommendation prompt for query
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7865 <> https://447c18ad4070dd2bb7.gradio.live
Killing tunnel 127.0.0.1:7866 <> https://1121f9bb675ff7c862.gradio.live
